In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import PowerTransformer

INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\normalized"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring"
METRIC_PATH = os.path.join(OUTPUT_PATH, "metricas_generacion")
os.makedirs(METRIC_PATH, exist_ok=True)

def add_features(df):
    df_extended = df.copy().astype(np.float32)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if df[col].nunique() > 2]

    detalle_columnas = []

    for col in numeric_cols:
        try:
            col_data = df[col].replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32)

            # Log, sqrt, squared
            df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_log", "log"))

            df_extended[f"{col}_sqrt"] = np.sqrt(np.abs(col_data))
            detalle_columnas.append((col, f"{col}_sqrt", "sqrt"))

            df_extended[f"{col}_squared"] = col_data ** 2
            detalle_columnas.append((col, f"{col}_squared", "squared"))

            # Yeo-Johnson
            pt_yj = PowerTransformer(method='yeo-johnson')
            yj_data = pt_yj.fit_transform(col_data.values.reshape(-1, 1)).flatten()
            df_extended[f"{col}_yeojohnson"] = yj_data
            detalle_columnas.append((col, f"{col}_yeojohnson", "yeojohnson"))

            # Box-Cox (solo si todos los valores son >0)
            if (col_data > 0).all():
                pt_bc = PowerTransformer(method='box-cox')
                bc_data = pt_bc.fit_transform(col_data.values.reshape(-1, 1)).flatten()
                df_extended[f"{col}_boxcox"] = bc_data
                detalle_columnas.append((col, f"{col}_boxcox", "boxcox"))

        except Exception as e:
            print(f"⚠️ Error en columna {col}: {e}")

    return df_extended, pd.DataFrame(detalle_columnas, columns=["original", "generada", "tipo"])

# Procesar cada subcarpeta
for norm_name in os.listdir(INPUT_PATH):
    norm_dir = os.path.join(INPUT_PATH, norm_name)
    if not os.path.isdir(norm_dir):
        continue  # Salta archivos, solo procesa carpetas
    out_dir = os.path.join(OUTPUT_PATH, norm_name)
    os.makedirs(out_dir, exist_ok=True)

    try:
        X_train = pd.read_parquet(f"{norm_dir}/X_train_{norm_name}.parquet")
        X_test = pd.read_parquet(f"{norm_dir}/X_test_{norm_name}.parquet")
        y_train = pd.read_parquet(f"{norm_dir}/y_train_{norm_name}.parquet")
        y_test = pd.read_parquet(f"{norm_dir}/y_test_{norm_name}.parquet")
    except Exception as e:
        print(f"❌ Error leyendo archivos en {norm_name}: {e}")
        continue

    # Guardar versión original
    X_train.to_parquet(f"{out_dir}/X_train_{norm_name}_ORIGINAL.parquet")
    X_test.to_parquet(f"{out_dir}/X_test_{norm_name}_ORIGINAL.parquet")
    y_train.to_parquet(f"{out_dir}/y_train_{norm_name}_ORIGINAL.parquet")
    y_test.to_parquet(f"{out_dir}/y_test_{norm_name}_ORIGINAL.parquet")

    # Aplicar feature engineering SOLO en train
    X_train_fe, df_detalle = add_features(X_train)

    # Replicar las columnas generadas en test
    X_test_fe = X_test.copy()
    for idx, row in df_detalle.iterrows():
        col = row["generada"]
        base_col = row["original"]
        tipo = row["tipo"]
        try:
            base_data = X_test[base_col].replace([np.inf, -np.inf], np.nan).fillna(0)
            if tipo == "log":
                X_test_fe[col] = np.log1p(np.abs(base_data))
            elif tipo == "sqrt":
                X_test_fe[col] = np.sqrt(np.abs(base_data))
            elif tipo == "squared":
                X_test_fe[col] = base_data ** 2
            elif tipo == "yeojohnson":
                pt_yj = PowerTransformer(method='yeo-johnson')
                X_test_fe[col] = pt_yj.fit(X_train[[base_col]].replace([np.inf, -np.inf], np.nan).fillna(0)).transform(base_data.values.reshape(-1, 1)).flatten()
            elif tipo == "boxcox":
                if (base_data > 0).all():
                    pt_bc = PowerTransformer(method='box-cox')
                    X_test_fe[col] = pt_bc.fit(X_train[[base_col]].replace([np.inf, -np.inf], np.nan).fillna(0)).transform(base_data.values.reshape(-1, 1)).flatten()
        except Exception as e:
            print(f"⚠️ Error generando {col} en test: {e}")

    # Asegurar mismo orden de columnas
    X_test_fe = X_test_fe[X_train_fe.columns]

    # Guardar nuevos datasets
    X_train_fe.to_parquet(f"{out_dir}/X_train_{norm_name}_FE.parquet")
    X_test_fe.to_parquet(f"{out_dir}/X_test_{norm_name}_FE.parquet")
    y_train.to_parquet(f"{out_dir}/y_train_{norm_name}_FE.parquet")
    y_test.to_parquet(f"{out_dir}/y_test_{norm_name}_FE.parquet")

    # Guardar CSV con el detalle de las columnas generadas
    detalle_file = os.path.join(METRIC_PATH, f"detalle_columnas_{norm_name}.csv")
    df_detalle.to_csv(detalle_file, index=False)

    # Métricas en consola
    resumen = df_detalle["tipo"].value_counts()
    print(f"\n📊 {norm_name} - Ingeniería aplicada")
    print(f"📌 Columnas originales: {X_train.shape[1]}")
    print(f"🧪 Nuevas columnas generadas: {df_detalle.shape[0]}")
    print(f"🧮 Total final de columnas: {X_train_fe.shape[1]}")
    print("🔍 Detalle por tipo de transformación:")
    for tipo, count in resumen.items():
        print(f"   ➤ {tipo}: {count}")
    print(f"📤 CSV guardado en: {detalle_file}")

c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:208: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_yeojohnson"] = yj_data
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_boxcox"] = 


📊 MaxAbs - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 617
🧮 Total final de columnas: 1156
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
   ➤ yeojohnson: 147
   ➤ boxcox: 29
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_MaxAbs.csv


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:208: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_yeojohnson"] = yj_data
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.


📊 MinMax - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 588
🧮 Total final de columnas: 1127
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
   ➤ yeojohnson: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_MinMax.csv


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:197: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:208: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:197: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\numpy\_core\_methods.py:208: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\scipy\stats\_morestats.py:1153: UserWarning: The optimal 


📊 NoNorm - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 617
🧮 Total final de columnas: 1156
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
   ➤ yeojohnson: 147
   ➤ boxcox: 29
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_NoNorm.csv


C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_yeojohnson"] = yj_data
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform


📊 Robust - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 588
🧮 Total final de columnas: 1127
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
   ➤ yeojohnson: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_Robust.csv


C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:35: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_yeojohnson"] = yj_data
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_extended[f"{col}_log"] = np.log1p(np.abs(col_data))
C:\Users\Gonzalo\AppData\Local\Temp\ipykernel_30668\76809867.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform


📊 Standard - Ingeniería aplicada
📌 Columnas originales: 539
🧪 Nuevas columnas generadas: 588
🧮 Total final de columnas: 1127
🔍 Detalle por tipo de transformación:
   ➤ log: 147
   ➤ sqrt: 147
   ➤ squared: 147
   ➤ yeojohnson: 147
📤 CSV guardado en: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring\metricas_generacion\detalle_columnas_Standard.csv
